# 03. Preprocessing & Splitting

Cleans raw text fields, performs stratified train/val/test splitting, and outputs processed datasets to prevent data leakage.

In [ ]:
import os
import re
import unicodedata
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

dataset_path = os.path.join("..", "..", "datasets", "raw", "udemy_courses.csv")
df = pd.read_csv(dataset_path)

def clean_text(text):
    if not text:
        return ""
    text = unicodedata.normalize("NFKD", str(text))
    text = text.lower()
    text = re.sub(r"<[^>]*>", " ", text)
    text = re.sub(r"https?://\S+|www\.\S+", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

df['cleaned_text'] = df['course_title'].apply(clean_text)

le = LabelEncoder()
df['encoded_label'] = le.fit_transform(df['subject'])

# Stratified Split (70% Train, 15% Val, 15% Test)
train_df, test_df = train_test_split(
    df, test_size=0.30, random_state=42, stratify=df['encoded_label']
)
val_df, test_df = train_test_split(
    test_df, test_size=0.50, random_state=42, stratify=test_df['encoded_label']
)

train_df['split'] = 'train'
val_df['split'] = 'val'
test_df['split'] = 'test'

processed_df = pd.concat([train_df, val_df, test_df])
output_dir = os.path.join("..", "..", "datasets", "processed")
os.makedirs(output_dir, exist_ok=True)
processed_df.to_csv(os.path.join(output_dir, "course_classification_processed.csv"), index=False)
print(f"Train size: {len(train_df)}, Val size: {len(val_df)}, Test size: {len(test_df)}")